# Kazakh Fact-Checking Benchmark — прогон в Google Colab

Онлайн-прогон без терминала. Colab имеет открытый интернет, поэтому внешние API доступны.

## Порядок действий
1. Слева нажми значок 🔑 (**Secrets**) и добавь ключи:
   - `GEMINI_API_KEY` — ключ Google AI Studio
   - `LLAMA_API_KEY` — ключ Groq (по желанию)
   Напротив каждого включи тумблер **Notebook access**.
2. Выполняй ячейки сверху вниз (Shift+Enter).

⚠️ Ключи храни ТОЛЬКО в Secrets. Никогда не вставляй их в ячейки и не коммить в репозиторий.

### 1. Клонируем репозиторий и ставим зависимости

In [ ]:
BRANCH = 'claude/kazakh-factcheck-benchmark-e76kd5'
REPO = 'github.com/Tim2190/kazakh-factcheck-benchmark.git'
!git clone --branch $BRANCH https://$REPO
%cd kazakh-factcheck-benchmark
!pip install -q -r requirements.txt
print('OK')

### 2. Загружаем ключи из Colab Secrets в окружение

In [ ]:
import os
from google.colab import userdata

for name in ['GEMINI_API_KEY', 'LLAMA_API_KEY']:
    try:
        val = userdata.get(name)
        if val:
            os.environ[name] = val
            print(f'{name}: загружен ({val[:4]}...)')
        else:
            print(f'{name}: пусто — пропускаем')
    except Exception:
        print(f'{name}: не найден в Secrets — пропускаем')

### 3. Прогон Gemini (батч-режим)
`--batch` шлёт весь документ + все тезисы **одним** запросом. Это нужно из-за лимита Gemini free **20 запросов/день (RPD)**; TPM 250K спокойно вмещает документ.

Модель по умолчанию — `gemini-2.5-flash`. Если у неё нет квоты (`429 / limit: 0`), поменяй `model_id` в `scripts/models.json` на доступную (напр. `gemini-3.1-flash-lite` — 500 RPD, или `gemma-4-31b`).

In [ ]:
!python scripts/export_dataset.py
!python scripts/run_factcheck.py --model gemini --batch
!python scripts/score.py results/gemini_leg_text01_run.json

### 4. Llama через Groq — внимание
Полный документ (~50k токенов) **не влезает** в бесплатный Groq (ошибка `413 Payload Too Large` / лимит TPM). Ячейка ниже это попробует, но, скорее всего, упадёт.

Если упадёт — прогони Llama **через веб-чат**: открой `prompts/chat_run_leg_text01.txt`, вставь в чат Llama (HuggingChat / Together Playground / groq.com playground), сохрани ответ-массив в `results/llama_leg_text01_run.json` и оцени последней ячейкой.

In [ ]:
!python scripts/run_factcheck.py --model llama --batch
!python scripts/score.py results/llama_leg_text01_run.json

### 5. Скачать результаты
Файлы в `results/` (сырые ответы + вердикты). Скачай и пришли мне — соберу итоговую таблицу и метрики (Wilson CI, McNemar).

In [ ]:
from google.colab import files
import glob
for f in glob.glob('results/*_run.json'):
    print('скачиваю', f)
    files.download(f)